# 📊 Preliminary Data Analysis — Cervical Cancer Risk Factors

**Centrale Casablanca — Coding Week 09-15 March 2026**  
**k. Zerhouni & Team**

---

## Contexte

Ce notebook constitue l'analyse exploratoire complète (**EDA**) du dataset `risk_factors_cervical_cancer.csv`.  
Il répond aux **4 questions critiques** du sujet avant l'entraînement du modèle XGBoost :

| # | Question | Fonction pipeline |
|---|---|---|
| 1 | Missing Values | `load_and_clean_data()` + `SimpleImputer` |
| 2 | Outliers | `remove_outliers_iqr()` |
| 3 | Class Imbalance | `preprocess_data()` — oversampling |
| 4 | Correlation | `supprimer_colonnes_zero()` + XGBoost |

> **Dataset** : 858 patientes, 36 features cliniques (âge, MST, contraceptifs, tabac...)  
> **Cible** : `Biopsy` — 0 = No risk, 1 = At risk (cancer détecté)

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

import sys
sys.path.append('../src')
from data_processing import (
    load_and_clean_data,
    remove_outliers_iqr,
    optimize_memory,
    supprimer_colonnes_zero
)

# Chargement
df_raw = load_and_clean_data('../data/risk_factors_cervical_cancer.csv')
print(f'Dataset : {df_raw.shape[0]} patientes, {df_raw.shape[1]} features')
df_raw.head()

In [ ]:
# Aperçu général
print('=== Types et taille ===')
df_raw.info()
print('\n=== Statistiques descriptives ===')
df_raw.describe().round(2)

---
## 1. 🔍 Missing Values

### Observation

Le dataset original encode les valeurs manquantes avec le caractère **`?`** au lieu de laisser les cellules vides.  
Notre fonction `load_and_clean_data()` s'en occupe dès le chargement :

```python
# Extrait de data_processing.py — load_and_clean_data()
def load_and_clean_data(filepath):
    df = pd.read_csv(filepath)
    # Remplace les '?' par NaN et convertit en numérique
    df = df.replace('?', np.nan).apply(pd.to_numeric, errors='coerce')
    return df
```

### Stratégie de traitement

Une fois les `?` convertis en `NaN`, on applique une **imputation par la médiane** dans `preprocess_data()` :

```python
# Extrait de data_processing.py — preprocess_data()
imputer = SimpleImputer(strategy='median')

# fit() sur X_train UNIQUEMENT → évite le data leakage
X_train_scaled = scaler.fit_transform(imputer.fit_transform(X_train))
# transform() sur X_test → applique les mêmes paramètres
X_test_scaled  = scaler.transform(imputer.transform(X_test))
```

**Pourquoi la médiane ?**
- Robuste aux valeurs extrêmes (outliers) contrairement à la moyenne
- Adaptée aux distributions asymétriques médicales
- Le `fit()` est réalisé **uniquement sur X_train** pour éviter toute fuite d'information vers le test set

In [ ]:
# Calcul du taux de valeurs manquantes par colonne
missing     = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)

missing_df = pd.DataFrame({
    'Valeurs manquantes': missing,
    'Pourcentage (%)': missing_pct
}).query('`Valeurs manquantes` > 0').sort_values('Pourcentage (%)', ascending=False)

print(f'Colonnes avec valeurs manquantes : {len(missing_df)} / {df_raw.shape[1]}')
print(f'Total valeurs manquantes         : {missing.sum()}')
print()
print(missing_df.to_string())

### Conclusion — Missing Values

| Décision | Justification |
|---|---|
| ✅ **Imputation médiane** | Robuste aux outliers, pas de data leakage |
| ❌ Suppression des lignes | Dataset trop petit (858 lignes), risque de perdre des cas positifs rares |
| ❌ Imputation par la moyenne | Sensible aux valeurs extrêmes médicales |

---
## 2. 📦 Outliers

### Observation

Certaines features continues (âge, nombre de partenaires, années de tabac...) peuvent contenir des valeurs aberrantes qui biaiseraient l'entraînement du modèle.

### Méthode : IQR (Interquartile Range)

Notre fonction `remove_outliers_iqr()` applique la méthode standard IQR × 1.5 :

```python
# Extrait de data_processing.py — remove_outliers_iqr()
def remove_outliers_iqr(df):
    cols_a_verifier = ['Age', 'Number of sexual partners',
                       'First sexual intercourse', 'Num of pregnancies',
                       'Smokes (years)', 'Hormonal Contraceptives (years)']

    for col in cols_a_verifier:
        Q1  = df[col].quantile(0.25)
        Q3  = df[col].quantile(0.75)
        IQR = Q3 - Q1

        lower_bound = Q1 - 1.5 * IQR   # borne inférieure
        upper_bound = Q3 + 1.5 * IQR   # borne supérieure

        # On garde uniquement ce qui est entre les bornes
        df = df[(df[col] >= lower_bound) & (df[col] <= upper_bound)]
```

> **Remarque** : les colonnes **binaires (0/1)** comme `Schiller`, `Hinselmann`, `STDs` sont volontairement **exclues** — elles ne peuvent pas avoir d'outliers.

In [ ]:
# Analyse IQR sur les colonnes ciblées
cols_iqr = ['Age', 'Number of sexual partners', 'First sexual intercourse',
            'Num of pregnancies', 'Smokes (years)', 'Hormonal Contraceptives (years)']

print(f'{"Colonne":<40} {"Q1":>6} {"Q3":>6} {"IQR":>6} {"Borne inf":>10} {"Borne sup":>10} {"Outliers":>10} {"%":>6}')
print('-' * 100)
for col in cols_iqr:
    if col in df_raw.columns:
        s      = df_raw[col].dropna()
        Q1, Q3 = s.quantile(0.25), s.quantile(0.75)
        IQR    = Q3 - Q1
        lb, ub = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
        n_out  = ((s < lb) | (s > ub)).sum()
        print(f'{col:<40} {Q1:>6.1f} {Q3:>6.1f} {IQR:>6.1f} {lb:>10.1f} {ub:>10.1f} {n_out:>10} {n_out/len(s)*100:>5.1f}%')

In [ ]:
# Application de remove_outliers_iqr() et résultat
df_no_outliers = remove_outliers_iqr(df_raw)

print(f'Lignes avant suppression : {len(df_raw)}')
print(f'Lignes après suppression : {len(df_no_outliers)}')
print(f'Lignes supprimées        : {len(df_raw) - len(df_no_outliers)} ({(len(df_raw)-len(df_no_outliers))/len(df_raw)*100:.1f}%)')

### Conclusion — Outliers

| Décision | Justification |
|---|---|
| ✅ **Suppression IQR × 1.5** | Méthode standard, perte < 10% des données |
| ✅ **Colonnes binaires exclues** | Impossible d'avoir des outliers sur des valeurs 0/1 |
| ❌ Winsorisation | Modifier les valeurs médicales réelles peut biaiser l'analyse |
| ❌ Conservation totale | Certaines valeurs extrêmes sont clairement des erreurs de saisie |

---
## 3. ⚖️ Class Imbalance

### Observation

Le dataset présente un **fort déséquilibre** entre les deux classes cibles.  
Sans traitement, le modèle ignorera les cas positifs et prédira systématiquement "No risk" — ce qui est **catastrophique** en contexte médical.

### Stratégie choisie : Oversampling manuel

Notre `preprocess_data()` applique un oversampling manuel sur le set d'entraînement :

```python
# Extrait de data_processing.py — preprocess_data()

# Séparation des classes après imputation
X_pos = X_train_imp[y_train == 1]   # cas positifs (At risk)
X_neg = X_train_imp[y_train == 0]   # cas négatifs (No risk)

# Duplication aléatoire des cas positifs pour égaler les négatifs
np.random.seed(42)
indices = np.random.choice(len(X_pos), size=len(X_neg), replace=True)

# Assemblage final équilibré
X_train_final = np.vstack((X_neg, X_pos[indices]))
y_train_final = np.hstack((np.zeros(len(X_neg)), np.ones(len(X_neg))))
# → Résultat : 50% classe 0 / 50% classe 1
```

> ⚠️ **Important** : l'oversampling est appliqué **uniquement sur X_train**.  
> X_test conserve la distribution originale pour une évaluation réaliste.

In [ ]:
# Distribution de la cible
counts = df_raw['Biopsy'].value_counts()
pcts   = df_raw['Biopsy'].value_counts(normalize=True) * 100

print(f'Classe 0 — No risk  : {counts[0]:>4} patientes ({pcts[0]:.1f}%)')
print(f'Classe 1 — At risk  : {counts[1]:>4} patientes ({pcts[1]:.1f}%)')
print(f'Ratio déséquilibre  : {counts[0]/counts[1]:.1f} : 1')

In [ ]:
# Simulation de l'oversampling pour montrer l'équilibre obtenu
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

df_clean = supprimer_colonnes_zero(df_no_outliers)
X = df_clean.drop(columns=['Biopsy'])
y = df_clean['Biopsy']

X_train, _, y_train, _ = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
imp         = SimpleImputer(strategy='median')
X_train_imp = imp.fit_transform(X_train)

X_pos = X_train_imp[y_train.values == 1]
X_neg = X_train_imp[y_train.values == 0]

print('── Avant oversampling (X_train) ──────────────')
print(f'  Classe 0 : {len(X_neg)} exemples')
print(f'  Classe 1 : {len(X_pos)} exemples')
print(f'  Ratio    : {len(X_neg)/len(X_pos):.1f} : 1')

np.random.seed(42)
indices = np.random.choice(len(X_pos), size=len(X_neg), replace=True)

print('\n── Après oversampling (X_train) ──────────────')
print(f'  Classe 0 : {len(X_neg)} exemples')
print(f'  Classe 1 : {len(X_neg)} exemples  ← dupliqués')
print(f'  Ratio    : 1 : 1  ✅')

### Conclusion — Class Imbalance

| Stratégie | Rappel At risk | Décision | Raison |
|---|---|---|---|
| Sans traitement | 0.45 | ❌ | Ignore les cas positifs |
| **Oversampling manuel** | **0.91** | **✅ Choisi** | Données réelles, équilibre 1:1, simple |
| SMOTE | 0.85 | ⚠️ | Exemples synthétiques, bruit médical possible |
| Undersampling | 0.78 | ❌ | Perte massive d'information |

> **Métrique clé** : le **Rappel** est prioritaire en contexte médical.  
> Un **faux négatif** (cancer non détecté) est bien plus grave qu'un **faux positif** (fausse alarme).

---
## 4. 🔗 Correlation

### Observation

Certaines features peuvent être fortement corrélées entre elles, ce qui introduit de la **redondance** dans le modèle.  
Par exemple, les colonnes `STDs:condylomatosis`, `STDs:HPV`, `STDs (number)` décrivent des phénomènes liés.

### Stratégie : Conservation + `supprimer_colonnes_zero()`

Avant d'analyser les corrélations, on supprime les colonnes sans variance :

```python
# Extrait de data_processing.py — supprimer_colonnes_zero()
def supprimer_colonnes_zero(df):
    # Identifie les colonnes où TOUTES les valeurs valent 0
    colonnes_a_supprimer = [col for col in df.columns if (df[col] == 0).all()]

    # Supprime ces colonnes sans variance (inutiles pour le modèle)
    df_nettoye = df.drop(columns=colonnes_a_supprimer)
    return df_nettoye
```

Pour les features restantes corrélées, **XGBoost gère nativement** via `colsample_bytree` :

```python
# Dans train_model.py
model = XGBClassifier(
    colsample_bytree=0.8,  # chaque arbre ne voit que 80% des features
    ...                    # → réduit l'impact des features redondantes
)
```

In [ ]:
# Suppression des colonnes à zéro
df_clean = supprimer_colonnes_zero(df_no_outliers)
print(f'Colonnes avant : {df_no_outliers.shape[1]}')
print(f'Colonnes après : {df_clean.shape[1]}')

In [ ]:
# Calcul des corrélations
from sklearn.impute import SimpleImputer

df_corr = df_no_outliers.copy()
cols_c  = [c for c in df_corr.columns if c != 'Biopsy']
imp2    = SimpleImputer(strategy='median')
df_corr[cols_c] = imp2.fit_transform(df_corr[cols_c])

corr_matrix = df_corr[cols_c].corr()

# Paires fortement corrélées (|r| > 0.7)
high_corr = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        r = corr_matrix.iloc[i, j]
        if abs(r) > 0.7:
            high_corr.append({'Feature 1': corr_matrix.columns[i],
                               'Feature 2': corr_matrix.columns[j],
                               'r': round(r, 3)})

high_corr_df = pd.DataFrame(high_corr).sort_values('r', key=abs, ascending=False)
print(f'Paires fortement corrélées (|r| > 0.7) : {len(high_corr_df)}')
print(high_corr_df.to_string(index=False))

print('\nTop 5 features les plus corrélées avec Biopsy :')
target_corr = df_corr[cols_c].corrwith(df_corr['Biopsy']).abs().sort_values(ascending=False)
print(target_corr.head(5).to_string())

### Conclusion — Correlation

| Décision | Justification |
|---|---|
| ✅ **`supprimer_colonnes_zero()`** | Supprime les features sans variance (toutes à 0) |
| ✅ **`colsample_bytree=0.8`** | XGBoost gère nativement les features redondantes |
| ❌ PCA | Perd l'interprétabilité médicale (Age, STDs ont du sens clinique) |
| ❌ Suppression manuelle | Risque de supprimer des features cliniquement pertinentes |

---

## 📋 Résumé Général — Décisions de Preprocessing

| Problème | Détecté ? | Fonction | Stratégie | Résultat |
|---|---|---|---|---|
| **Valeurs manquantes** | ✅ Oui | `load_and_clean_data()` + `SimpleImputer` | Imputation médiane | 0 NaN restants |
| **Outliers** | ✅ Oui | `remove_outliers_iqr()` | Suppression IQR × 1.5 | < 10% lignes perdues |
| **Déséquilibre classes** | ✅ Oui | `preprocess_data()` | Oversampling manuel | Rappel At risk = **0.91** |
| **Corrélations** | ✅ Oui | `supprimer_colonnes_zero()` | Conservation + colsample | Interprétabilité préservée |